# Lab 3 - writing a gradient descent loop

**Session 3.** Implement `fit_gd` yourself, watch the loss curve, and confirm it reaches
the closed-form solution from session 2. Then break it deliberately.

## 1. Data, standardised

In [ ]:
# The twelve flats from the lectures. The same rows are released as
# data/housing-mini.csv in the cohort materials repo.
import numpy as np

area = np.array([32, 45, 52, 60, 68, 75, 80, 95, 38, 55, 110, 48], float)
dist = np.array([0.3, 0.9, 0.4, 1.6, 0.7, 2.1, 1.1, 0.5, 1.8, 0.6, 1.4, 2.6])
rent = np.array([540, 510, 640, 545, 720, 620, 770, 860, 420, 640, 930, 400], float)
print(area.shape, dist.shape, rent.shape)

In [ ]:
def standardise(x):
    """z-scores plus the statistics, so the same transform can be reapplied."""
    mu, sd = x.mean(), x.std()
    return (x - mu) / sd, mu, sd


X_raw = np.column_stack([area, dist])
X = np.column_stack([np.ones(len(area))] + [standardise(col)[0] for col in X_raw.T])
y = rent
print(X.round(3)[:4])

## 2. The loss and its gradient

`L(b) = (1/n)||y - Xb||^2` and `grad L(b) = -(2/n) X'(y - Xb)`. Two lines, and they are the
whole algorithm.

In [ ]:
def loss(X, y, b):
    resid = y - X @ b
    return float(resid @ resid / len(y))


def grad(X, y, b):
    resid = y - X @ b
    return -2.0 / len(y) * (X.T @ resid)

## 3. `fit_gd`

In [ ]:
def fit_gd(X, y, alpha=0.1, n_iter=500, tol=1e-10):
    """Batch gradient descent. Returns (coefficients, loss history)."""
    b = np.zeros(X.shape[1])
    history = [loss(X, y, b)]
    for _ in range(n_iter):
        b = b - alpha * grad(X, y, b)
        history.append(loss(X, y, b))
        if abs(history[-2] - history[-1]) < tol * max(1.0, history[-2]):
            break                      # relative improvement below tolerance
    return b, np.array(history)


b_gd, hist = fit_gd(X, y, alpha=0.1, n_iter=2000)
print(f"converged after {len(hist) - 1} iterations, final loss {hist[-1]:.4f}")
print("coefficients:", b_gd.round(4))

## 4. Does it agree with the closed form?

In [ ]:
b_exact, *_ = np.linalg.lstsq(X, y, rcond=None)
print("closed form :", b_exact.round(4))
print("descent     :", b_gd.round(4))
print("max abs diff:", np.abs(b_exact - b_gd).max())

## 5. The loss curve

Always plot this. A learning rate is not tuned until you have looked at the curve.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 3.5))
for alpha in (0.01, 0.05, 0.1, 0.3):
    _, h = fit_gd(X, y, alpha=alpha, n_iter=200, tol=0.0)
    ax.plot(h, label=f"alpha = {alpha}")
ax.set_yscale("log")
ax.set_xlabel("iteration")
ax.set_ylabel("mean squared error (log scale)")
ax.set_title("Loss per iteration")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Break it

Descent diverges when the step is large relative to the curvature of the loss. Find the
boundary.

In [ ]:
for alpha in (0.5, 0.9, 1.0, 1.1, 2.0):
    _, h = fit_gd(X, y, alpha=alpha, n_iter=100, tol=0.0)
    verdict = "converged" if np.isfinite(h[-1]) and h[-1] < h[0] else "DIVERGED"
    print(f"alpha = {alpha:<4} final loss = {h[-1]:>14.4g}   {verdict}")

## Exercises

1. **Scaling matters.** Re-run `fit_gd` on the *unstandardised* design matrix
   (`np.column_stack([np.ones(12), area, dist])`) at `alpha = 0.1`. What happens, and why?
   Then find an `alpha` that does converge, and compare how many iterations it needs.
2. **Stochastic descent.** Write `fit_sgd` that uses one random row per step. Plot its loss
   curve against the batch curve on the same axes. The noise is not a defect - explain in one
   sentence what it buys you in session 8.
3. **Early stopping.** Hold back four rows, and stop when the *validation* loss stops
   improving rather than the training loss. How many iterations does it use?